<a href="https://colab.research.google.com/github/look4pritam/NetflixPrize/blob/master/Notebooks/v1.0.0/v1.0.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Netflix Prize

## Install Python packages.

In [1]:
!pip install numpy pandas scipy tqdm

In [2]:
!pip install implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 7.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=10906382 sha256=b793cd07d71a5fdf50ef3449751ee0bd80ff6b3aef61816b01c797258a63b4b0
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit


## Set the project root directory.

In [3]:
import os

root_dir = '/content/'
os.chdir(root_dir)

!ls -al

total 20
drwxr-xr-x 1 root root 4096 Apr 21 06:06 .
drwxr-xr-x 1 root root 4096 Apr 21 05:50 ..
drwxr-xr-x 4 root root 4096 Apr 16 13:33 .config
-rw-r--r-- 1 root root   67 Apr 21 06:06 kaggle.json
drwxr-xr-x 1 root root 4096 Apr 16 13:33 sample_data


## Download the Netflix Prize dataset from Kaggle.

### Set the Kaggle API key.

### Generate and upload 'kaggle.json' file.

In [4]:
os.environ['KAGGLE_CONFIG_DIR'] = root_dir
!chmod 600 /content/kaggle.json

### Download the Netflix Prize dataset.

In [5]:
!kaggle datasets download netflix-inc/netflix-prize-data
!ls -al

Dataset URL: https://www.kaggle.com/datasets/netflix-inc/netflix-prize-data
License(s): other
100% 683M/683M [00:44<00:00, 16.1MB/s]

total 699436
drwxr-xr-x 1 root root      4096 Apr 21 06:15 .
drwxr-xr-x 1 root root      4096 Apr 21 05:50 ..
drwxr-xr-x 4 root root      4096 Apr 16 13:33 .config
-rw------- 1 root root        67 Apr 21 06:06 kaggle.json
-rw-r--r-- 1 root root 716193814 Nov 13  2019 netflix-prize-data.zip
drwxr-xr-x 1 root root      4096 Apr 16 13:33 sample_data


In [6]:
import os
import zipfile

### Extract the Netflix Prize dataset.

In [7]:
zip_path = "netflix-prize-data.zip"
dataset_path = "netflix_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(dataset_path)

print("Extracted Netflix Prize dataset from:", os.listdir(dataset_path))

Extracted Netflix Prize dataset from: ['combined_data_2.txt', 'README', 'movie_titles.csv', 'probe.txt', 'combined_data_3.txt', 'combined_data_4.txt', 'combined_data_1.txt', 'qualifying.txt']


## Load the Netflix Prize dataset.

### Import Python packages.

In [8]:
import numpy as np
import pandas as pd
from tqdm import tqdm

In [9]:
from scipy.sparse import coo_matrix

In [10]:
def build_sparse_matrix(input_folder):

    user_ids = []
    movie_ids = []
    ratings = []

    user_map = {}
    movie_map = {}

    user_counter = 0
    movie_counter = 0

    files = [
        "combined_data_1.txt",
        "combined_data_2.txt",
        "combined_data_3.txt",
        "combined_data_4.txt"
    ]

    for file in files:

        path = os.path.join(input_folder, file)

        print("\nProcessing input file: ", file)

        movie_id = None

        with open(path, "r", encoding="latin-1") as f:

            for line in tqdm(f):

                line = line.strip()

                if not line:
                    continue

                # movie ID line
                if line.endswith(":"):

                    movie = int(line[:-1])

                    if movie not in movie_map:
                        movie_map[movie] = movie_counter
                        movie_counter += 1

                    movie_id = movie_map[movie]

                else:

                    parts = line.split(",")

                    if len(parts) < 2:
                        continue

                    user = int(parts[0])
                    rating = float(parts[1])

                    if user not in user_map:
                        user_map[user] = user_counter
                        user_counter += 1

                    user_ids.append(user_map[user])
                    movie_ids.append(movie_id)
                    ratings.append(rating)

    rating_matrix = coo_matrix(
        (ratings, (user_ids, movie_ids)),
        shape=(user_counter, movie_counter)
    )

    return rating_matrix.tocsr(), user_map, movie_map

In [11]:
rating_matrix, user_map, movie_map = build_sparse_matrix("netflix_dataset")

print("Matrix shape: ", rating_matrix.shape)
print("Number of ratings: ", rating_matrix.nnz)


Processing input file:  combined_data_1.txt


24058263it [00:34, 707547.24it/s]



Processing input file:  combined_data_2.txt


26982302it [00:38, 709443.58it/s]



Processing input file:  combined_data_3.txt


22605786it [00:30, 729395.89it/s]



Processing input file:  combined_data_4.txt


26851926it [00:38, 701938.84it/s]


Matrix shape:  (480189, 17770)
Number of ratings:  100480507


## Use Global Normalization.  

In [12]:
rows, cols = rating_matrix.nonzero()
true_ratings = rating_matrix.data

### Calculate global mean value.

In [13]:
global_mean = np.mean(true_ratings)
print(global_mean)

3.604289964420661


### Normalize dataset using global mean value.

In [14]:
rating_matrix.data = rating_matrix.data - global_mean

## Factorize matrix using Alternating Least Squares (ALS).

### Define alpha value.

In [15]:
alpha = 40

### Create confidence matrix.

In [16]:
C = rating_matrix.copy()
C.data = 1 + alpha * C.data

### Transpose for implicit ALS.

In [17]:
C = C.T.tocsr()

### Import 'AlternatingLeastSquares' algorithm.

In [18]:
from implicit.als import AlternatingLeastSquares

In [ ]:
model = AlternatingLeastSquares(
    factors=100,
    regularization=0.1,
    iterations=20,
    use_gpu=True
)

model.fit(C)

## Compute normalized RMSE metrics.

In [20]:
user_factors = model.item_factors.to_numpy()  ## users
item_factors = model.user_factors.to_numpy()  ## items

In [21]:
rows, cols = rating_matrix.nonzero()
true_ratings = rating_matrix.data

batch_size = 1_000_000
sq_error = 0.0
n = len(true_ratings)

for start in range(0, n, batch_size):
    end = min(start + batch_size, n)

    r = rows[start:end]
    c = cols[start:end]

    preds = np.sum(
        user_factors[r] * item_factors[c],
        axis=1
    )

    sq_error += np.sum((true_ratings[start:end] - preds) ** 2)

rmse_norm = np.sqrt(sq_error / n)
print("Normalized RMSE: ", rmse_norm)

Normalized RMSE:  0.9871618237417739


## Compute RMSE metrics.

In [23]:
rows, cols = rating_matrix.nonzero()
true_ratings = rating_matrix.data

batch_size = 1_000_000
sq_error = 0.0
n = len(true_ratings)

for start in range(0, n, batch_size):
    end = min(start + batch_size, n)

    r = rows[start:end]
    c = cols[start:end]

    preds = global_mean + np.sum(
        user_factors[r] * item_factors[c],
        axis=1
    )

    preds = np.clip(preds, 1, 5)

    original_ratings = true_ratings[start:end] + global_mean

    sq_error += np.sum((original_ratings - preds) ** 2)

rmse = np.sqrt(sq_error / n)
print("RMSE:", rmse)

RMSE: 0.9871416939334685
